# Name That LLM: offline submission (batch inference)

This notebook produces our competition submission. When you submit it, Kaggle **re-runs it
on ~4,000 hidden passages with the internet turned off**. While you edit it, it only sees 3
example test rows. The code must therefore work offline and must not assume a row count.

- **Code:** the project's package, attached as the dataset `ryqn00/name-that-llm-lib` and
  published automatically by CI on each version tag
  ([repo](https://github.com/Ryqn0/name-that-llm)).
- **Model, an A/B test:** with `ENSEMBLE = "ensemble_pool"` (I-029), an ensemble of three
  models, blended with weights 0.75 / 0.10 / 0.15:
  1. the TF-IDF text model `lr`, trained on our passages **plus 30,000 public generated
     passages** (the pool). On its own, without the pool, it scored public LB 0.320;
  2. a logistic regression on **Qwen 3 4B's hidden states** (layers 6 to 9, averaged over
     each passage's tokens). Blended with `lr`, it's our champion `lm_blend`: public LB 0.389;
  3. **ModernBERT-base, fine-tuned** to name the model: the average of its 5 fold models.
- **Why it might win:** on fresh cross-validation folds, the pool lifted the ensemble by
  +0.049 on seed 42 (CV 0.4705 against 0.4219; 95% CI [+0.032, +0.065]), and by +0.046 and
  +0.039 on seeds 43 and 44 (`scripts/pool_ensemble_check.py` in the repo). The LM head and
  the encoder are the same as in the 0.410 submission, so the leaderboard difference
  measures what the pool brings to `lr`. `ENSEMBLE = "ensemble"` gives back the 0.410
  submission; `USE_ENCODER = False`, `lm_blend` (0.389).
- **The pool:** the attached public dataset `crtproduction/name-that-llm-open-pool`,
  rebuilt here offline with the project's safeguards: open licenses only, copies of our
  passages and conflicting labels removed, and only the 5 sources that helped `lr`. `lr`
  trains on a seeded sample of 30,000 of its passages.
- **On the GPU:** Qwen 3 4B is attached as a Kaggle Model
  (`qwen-lm/qwen-3/transformers/4b-base`, Apache 2.0) and runs in float16. The encoder's 5
  fold models (about 1.5 GB) are the encoder-final notebook's output, attached here. torch
  and transformers come preinstalled in Kaggle's GPU image, so our package doesn't depend
  on them.

In [ ]:
# Internet is OFF, so install from the attached dataset. Search for the wheel instead of
# hard-coding the mount path: Kaggle's input layout has changed before (the competition data
# now sits under /kaggle/input/competitions/...).
import glob
import os

found = glob.glob("/kaggle/input/**/name_that_llm-*.whl", recursive=True)
wheel_dirs = {os.path.dirname(p) for p in found}
assert len(wheel_dirs) == 1, f"expected exactly one wheel folder, found: {wheel_dirs}"
WHEELS = wheel_dirs.pop()
print("installing from", WHEELS, "->", sorted(os.listdir(WHEELS)))

In [ ]:
# --no-index: never reach for PyPI (there's no internet). --find-links: take wheels from
# the attached folder. Packages Kaggle already has (pandas, scikit-learn, ...) stay as they are.
%pip install -q --no-index --find-links {WHEELS} name-that-llm

In [ ]:
import time

import numpy as np
import pandas as pd

import name_that_llm
from name_that_llm.config import BlendConfig, EncoderConfig, EnsembleConfig, load_config
from name_that_llm.data import load_test, load_train
from name_that_llm.submit import (
    predict_submission,
    predict_submission_blend,
    predict_submission_ensemble,
)

CHAMPION = "lr"  # the text-only champion: public LB 0.320
USE_LM = True  # P3b: blend lr with Qwen 3 hidden states (public LB 0.389). False = text-only lr
USE_ENCODER = True  # P5 A/B test: add the fine-tuned encoder. False = the P3b blend
BLEND = "lm_blend"  # the BlendConfig chosen by the sweep notebook (05-lm-sweep)
ENSEMBLE = "ensemble_pool"  # I-029: lr also trains on the pool. "ensemble": LB 0.410
assert USE_LM or not USE_ENCODER, "the ensemble blends the LM head too: set USE_LM = True"

ens_cfg = load_config(ENSEMBLE, model_cls=EnsembleConfig) if USE_ENCODER else None
enc_cfg = load_config(ens_cfg.encoder, model_cls=EncoderConfig) if USE_ENCODER else None
# Each config names the one it was tuned with: the ensemble its blend, the blend its text model.
blend_cfg = (
    load_config(ens_cfg.blend if USE_ENCODER else BLEND, model_cls=BlendConfig) if USE_LM else None
)
text_cfg = load_config(blend_cfg.text_config if USE_LM else CHAMPION)
print("package", name_that_llm.__version__, "| text model:", text_cfg.name)
print("blend:", blend_cfg)
print("ensemble:", ens_cfg)

train, test = load_train(), load_test()  # validated: ids stay strings, labels are checked
print("train", train.shape, "| test", test.shape)

# I-029: the text model also trains on the pool, rebuilt here offline from the attached
# public dataset, with the same safeguards as on the PC (pool.build_pool). Counts only.
pool = None
if USE_ENCODER and ens_cfg.text_pool:
    from pathlib import Path

    from name_that_llm.pool import build_pool, load_pool_sources

    pool = build_pool(load_pool_sources("pool"), Path("/kaggle/input"), train)
    print(f"pool: {len(pool):,} passages; lr trains on {min(ens_cfg.text_pool, len(pool)):,}")

## LM features on the GPU

Each passage becomes one vector: the average, over its tokens, of the model's hidden states
in the blend's layers. We extract them for train **and** test, here, in the same run.

**Why recompute the training features** instead of attaching the ones the
[features notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-features) saved? It
costs about 6 GPU-minutes, but it's one artifact less to keep in sync: train and test
features come from the same code, model and run, so they can't drift apart.

In [ ]:
extract_min = 0.0  # training-feature extraction time, added to the runtime check below
if USE_LM:
    import torch
    import transformers

    from name_that_llm.lm import find_model_dir, load_lm, pooled_hidden_states, select_layers

    print("torch", torch.__version__, "| transformers", transformers.__version__)
    # Fail in the first seconds, not after hours: the blend needs the GPU (accelerator: T4).
    assert torch.cuda.is_available(), "USE_LM = True needs a GPU"
    model, tokenizer = load_lm(find_model_dir(blend_cfg.lm_variation))  # searched, never hard-coded

    # Keep only the layers the blend uses (hidden_states index: 0 = embeddings, k = block k).
    # The forward pass computes every layer anyway; this just saves memory.
    LAYERS = range(blend_cfg.layer, blend_cfg.layer + blend_cfg.layer_window)

    def lm_features(texts):
        feats = pooled_hidden_states(
            texts,
            model,
            tokenizer,
            layers=LAYERS,
            batch_size=blend_cfg.batch_size,
            max_length=blend_cfg.max_length,
        )
        return select_layers(feats, 0, blend_cfg.layer_window)  # the mean of those layers

    start = time.time()
    lm_train = lm_features(train["text"].tolist())
    extract_min = (time.time() - start) / 60
    # STRICT on train: these are known passages, finite in the features notebook. A failure
    # here is a real problem, so stop before submitting anything.
    assert np.isfinite(lm_train).all(), "non-finite train features (see 04-lm-features)"

    lm_test = lm_features(test["text"].tolist())
    # LENIENT on test: float16 can overflow on an unusual passage, and one bad passage should
    # cost one prediction, not crash the 9-hour run. The TRAIN MEAN is the neutral input:
    # once scaled it's all zeros, so the LM head gives near-uniform probabilities and the
    # text model decides that passage.
    bad = ~np.isfinite(lm_test).all(axis=1)
    if bad.any():
        print(f"warning: {bad.sum()} test passages had non-finite LM features")
        lm_test[bad] = lm_train.mean(axis=0)
    print(f"LM features: train {lm_train.shape} in {extract_min:.1f} min, test {lm_test.shape}")

## The fine-tuned encoder on the GPU

The [encoder-final notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-encoder-final)
fine-tuned ModernBERT-base once per cross-validation fold and saved the 5 models. Nothing is
trained here: the encoder's prediction is the **average of those 5 models**, the same ones
whose out-of-fold predictions chose the ensemble's weights.

In [ ]:
if USE_ENCODER:
    from name_that_llm.encoder import find_fold_dirs, predict_proba_folds

    # Searched, never hard-coded, and exactly 5 or an error: averaging 4 models would
    # silently change the submission.
    FOLD_DIRS = find_fold_dirs(name="encoder", n_folds=5)
    print("fold models:", [str(d) for d in FOLD_DIRS])

    def encoder_proba(texts):
        # The training max_length, so every passage is cut exactly as it was in CV.
        p = predict_proba_folds(
            FOLD_DIRS, texts, batch_size=2 * enc_cfg.batch_size, max_length=enc_cfg.max_length
        )
        # LENIENT, as with the LM features: one bad passage costs one prediction, not the
        # run. A uniform row adds the same amount to all 18 labels, so it can't change which
        # label wins: the other two models decide that passage.
        bad = ~np.isfinite(p).all(axis=1)
        if bad.any():
            print(f"warning: {bad.sum()} passages had non-finite encoder probabilities")
            p[bad] = 1 / p.shape[1]
        return p

    start = time.time()
    p_encoder = encoder_proba(test["text"].tolist())
    print(f"encoder: test {p_encoder.shape} in {(time.time() - start) / 60:.1f} min")

In [ ]:
start = time.time()
# Fits on ALL of train (CV was only for measuring), predicts test in order, and VALIDATES
# the result (every id exactly once, no nulls, only the 18 labels) before we write anything.
if USE_ENCODER:
    sub = predict_submission_ensemble(
        text_cfg, blend_cfg, ens_cfg, train, test, lm_train, lm_test, p_encoder, pool=pool
    )
elif USE_LM:
    sub = predict_submission_blend(text_cfg, blend_cfg, train, test, lm_train, lm_test)
else:
    sub = predict_submission(text_cfg, train, test)
sub.to_parquet("/kaggle/working/submission.parquet", index=False)
print(f"wrote submission.parquet: {len(sub)} rows in {time.time() - start:.0f}s")
sub.head()

## Runtime check

The hidden rerun has ~4,000 passages and a **9-hour** limit. On 4,000 stand-in rows
(training texts, resampled) we time the LM extraction, the encoder's 5 models and a full fit
plus prediction, then add the training-feature extraction measured above. The total must be
far below that limit.

In [ ]:
stand_in = pd.DataFrame(
    {
        "id": [f"t{i}" for i in range(4000)],
        "text": train["text"].sample(4000, replace=True, random_state=0).to_numpy(),
    }
)
start = time.time()
if USE_LM:
    lm_stand_in = lm_features(stand_in["text"].tolist())
if USE_ENCODER:
    p_stand_in = encoder_proba(stand_in["text"].tolist())
    predict_submission_ensemble(
        text_cfg, blend_cfg, ens_cfg, train, stand_in, lm_train, lm_stand_in, p_stand_in, pool=pool
    )
elif USE_LM:
    predict_submission_blend(text_cfg, blend_cfg, train, stand_in, lm_train, lm_stand_in)
else:
    predict_submission(text_cfg, train, stand_in)
minutes = extract_min + (time.time() - start) / 60
print(f"estimated hidden run: {minutes:.1f} min (limit: 540 min)")
assert minutes < 120, "far too slow for comfort; investigate before submitting"